# 店舗売上需要予測: 実験 #004 33ファミリー別個別モデル ＆ ゼロ売上フィルター

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** 全商品を1つのモデルで解く限界を打破し、**33商品ファミリーごとに独立したLightGBMモデルを構築** ＋ **直近21日間売上ゼロ系列の完全0予測フィルター** による本番スコアブレイクスルー
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **モデル:** **33 Family-wise LightGBM Regressors**
- **現在自己ベスト:** **LB 0.52338**

---

## 1. なぜ商品ファミリー別モデルが最強なのか？

1. **スケール格差の解消:**
   - 1日数千本売れる `BEVERAGES`（飲料水）と、1日0〜1個の `BOOKS` や `BABY CARE` を1つのモデルで学習させると、巨大なカテゴリに決定木が引っ張られ、小規模カテゴリの予測が狂います。
2. **商品ごとの固有トレンドの学習:**
   - 8月後半の新学期に爆発する `SCHOOL AND OFFICE SUPPLIES` や、週末に集中する生鮮食品など、商品ごとの特性に特化した33個の専用モデルを学習させます。
3. **直近21日間売上ゼロ系列の完全0クリップ (Zero Forecast Filter):**
   - 1,782系列中、直近21日間に全く売上がない124系列（約2,000行）を完全に `0.0` に固定することで、微小予測値によるRMSLEペナルティを完全シャットアウトします。

## 2. ライブラリ読み込み & データ読み込み

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# データディレクトリ自動検出
def find_data_dir():
    candidates = [
        '/kaggle/input/store-sales-time-series-forecasting',
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '003_store-sales/data',
        '../data',
        'data'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    matches_local = glob.glob('**/train.csv', recursive=True)
    if matches_local:
        return os.path.dirname(matches_local[0])
    if os.path.exists('/kaggle/input'):
        input_contents = os.listdir('/kaggle/input')
        raise FileNotFoundError(f'/kaggle/input にデータがありません。現在の中身: {input_contents}。右上の [+ Add Input] から "store-sales-time-series-forecasting" を追加してください。')
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

# 直近21日間（2017-07-25 〜 2017-08-15）で売上が完全に0のペアを特定
recent_train = train_df[train_df['date'] >= '2017-07-25']
recent_sum = recent_train.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
zero_pairs = set(zip(recent_sum[recent_sum['sales'] == 0]['store_nbr'], recent_sum[recent_sum['sales'] == 0]['family']))
print(f'直近21日間で売上ゼロの (store_nbr, family) ペア数: {len(zero_pairs)} / 1,782 ペア')

## 3. カレンダー ＆ ラグ特徴量エンジニアリング

In [ ]:
test_df['sales'] = np.nan
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df['log_sales'] = np.log1p(all_df['sales'])

all_df['year'] = all_df['date'].dt.year
all_df['month'] = all_df['date'].dt.month
all_df['day'] = all_df['date'].dt.day
all_df['dayofweek'] = all_df['date'].dt.dayofweek
all_df['is_weekend'] = all_df['dayofweek'].isin([5, 6]).astype(int)
all_df['dayofyear'] = all_df['date'].dt.dayofyear
all_df['weekofyear'] = all_df['date'].dt.isocalendar().week.astype(int)
all_df['is_month_start'] = all_df['date'].dt.is_month_start.astype(int)
all_df['is_month_end'] = all_df['date'].dt.is_month_end.astype(int)
all_df['is_wage_day'] = ((all_df['day'] == 15) | (all_df['is_month_end'] == 1)).astype(int)
all_df['is_wage_day_after'] = ((all_df['day'] == 16) | (all_df['is_month_start'] == 1)).astype(int)

# ラグ特徴量作成
all_df = all_df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
group = all_df.groupby(['store_nbr', 'family'])['log_sales']

for lag in [16, 21, 28, 35]:
    all_df[f'sales_lag_{lag}'] = group.shift(lag)

lag16 = group.shift(16)
all_df['sales_rolling_mean_7'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
all_df['sales_rolling_mean_14'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(14, min_periods=1).mean()
)
all_df['sales_rolling_mean_28'] = lag16.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(28, min_periods=1).mean()
)

all_df['store_nbr'] = all_df['store_nbr'].astype('category')

train_feat = all_df[all_df['date'] <= '2017-08-15'].copy()
test_feat = all_df[all_df['date'] >= '2017-08-16'].copy()

features = [
    'store_nbr', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'is_weekend',
    'dayofyear', 'weekofyear', 'is_month_start', 'is_month_end', 'is_wage_day', 'is_wage_day_after',
    'sales_lag_16', 'sales_lag_21', 'sales_lag_28', 'sales_lag_35',
    'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28'
]
print(f'特徴量数: {len(features)} 列')

## 4. 33商品ファミリーごとの個別LightGBMモデル学習

In [ ]:
val_start_date = '2017-07-31'
val_end_date = '2017-08-15'
train_start_date = '2016-01-01'

trn_mask = (train_feat['date'] >= train_start_date) & (train_feat['date'] < val_start_date)
val_mask = (train_feat['date'] >= val_start_date) & (train_feat['date'] <= val_end_date)

families = train_df['family'].unique()
print(f'33商品ファミリーごとに独立したLightGBMモデルを順次学習中...')

val_preds = []
val_trues = []
test_predictions = []

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'random_state': 42,
    'verbose': -1,
    'n_estimators': 500,
}

for fam in families:
    fam_trn_mask = trn_mask & (train_feat['family'] == fam)
    fam_val_mask = val_mask & (train_feat['family'] == fam)
    fam_tst_mask = (test_feat['family'] == fam)

    X_trn = train_feat.loc[fam_trn_mask, features]
    y_trn = np.log1p(train_feat.loc[fam_trn_mask, 'sales'].values)
    X_v = train_feat.loc[fam_val_mask, features]
    y_v = train_feat.loc[fam_val_mask, 'sales'].values
    X_tst = test_feat.loc[fam_tst_mask, features]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_trn,
        y_trn,
        eval_set=[(X_v, np.log1p(y_v))],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )

    # 検証用予測
    pred_v = np.clip(np.expm1(model.predict(X_v)), 0, None)
    val_preds.extend(pred_v)
    val_trues.extend(y_v)

    # テスト予測 & ゼロフィルター適用
    pred_tst = np.clip(np.expm1(model.predict(X_tst)), 0, None)
    tst_stores = test_feat.loc[fam_tst_mask, 'store_nbr'].values
    for i, st in enumerate(tst_stores):
        if (st, fam) in zero_pairs:
            pred_tst[i] = 0.0

    fam_test_df = test_feat.loc[fam_tst_mask, ['id']].copy()
    fam_test_df['sales'] = pred_tst
    test_predictions.append(fam_test_df)

overall_rmsle = np.sqrt(mean_squared_error(np.log1p(val_trues), np.log1p(val_preds)))
print('=' * 65)
print(f'  🏆 [実験 #004] 33ファミリー別モデル 全体手元検証 RMSLE: {overall_rmsle:.5f}')
print('=' * 65)

## 5. テストデータ予測 & 提出ファイル作成 (id順ソート)

In [ ]:
# 全ファミリーの予測を結合し、id 順に整列
final_sub = pd.concat(test_predictions, ignore_index=True)
final_sub = final_sub.sort_values('id').reset_index(drop=True)
final_sub.to_csv('submission.csv', index=False)

print('提出用 submission.csv を作成しました（id順ソート完了）！')
print(f'行数: {len(final_sub):,} 行')
print(final_sub.head(10))
print(final_sub.describe())